# Find the failing RAG stage before you change the prompt

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/diagnose-retrieval-before-changing-the-generation-prompt/retrieval_v1.ipynb)

Companion to https://profrod.ai/articles/diagnose-retrieval-before-changing-the-generation-prompt. A four-stage research assistant runs on BEIR's SciFact (300 claims, 5,183 abstracts):
- retrieval with BM25 or MiniLM;
- reranking with or without a cross-encoder;
- a 1,024-token prompt budget;
- Qwen2.5 1.5B naming the abstract that bears on each claim.

Every miss is attributed to the first stage that lost the evidence.

Part 1 recomputes the article's numbers from the recorded per-claim lists. Part 2 reruns the pipeline on the first 60 claims on this runtime. Choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import json
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/diagnose-retrieval-before-changing-the-generation-prompt/"
FILES = ["retrieval_stats_v1.py", "retrieval_run_v1.py", "revisions-v1.json", "results/queries-v1.jsonl", "results/variant-v1.jsonl",
         "receipts/retrieval-v1.json"]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import retrieval_run_v1 as R

summary = R.summarize()
n = summary["queries"]
for key, p in summary["pipelines"].items():
    stages = {s: v["k"] for s, v in p["stages"].items()}
    print(f"{key:22s} correct {stages['none']:3d}/{n}   lost in: retrieval {stages['retrieval']}, reranking {stages['reranking']}, "
          f"prompt assembly {stages['prompt_assembly']}, generation {stages['generation']}")

In [ ]:
# The follow-up: only the generation prompt changes, on the same saved prompts.
v = summary["revisedPrompt"]
print({s: x["k"] for s, x in v["stages"].items()}, v["vsOriginal"])

## Part 2: rerun the first 60 claims here (optional, no key needed)

This downloads SciFact, checks its md5 against BEIR's, and runs all four pipelines on 60 claims. Expect a few minutes on a T4.

In [ ]:
!pip -q install sentence-transformers
os.makedirs("mine", exist_ok=True)
!cp retrieval_stats_v1.py retrieval_run_v1.py revisions-v1.json mine/
!cd mine && python retrieval_run_v1.py all --limit 60 && python retrieval_run_v1.py summarize > /dev/null
mine = json.load(open("mine/results/summary-v1.json"))
for key, p in mine["pipelines"].items():
    print(key, {s: v["k"] for s, v in p["stages"].items()})